# 04 · 特征工程与训练/测试集划分

目标是建立一个**通话前客户筛选**数据集，因此只使用拨打本次电话前可获得的信息。

## 特征策略

- 排除 `duration`：它是本次通话结束后才能得到的结果信息。
- 将 `pdays=999` 转换为“此前未联系”的业务含义。
- 构造年龄段、是否曾联系、是否有住房/个人贷款等特征。
- 训练/测试集按目标变量分层抽样，测试集只用于最终泛化评估。

In [1]:
from pathlib import Path
import sys

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.features import build_precall_features, stratified_train_test_split

DATA_PATH = PROJECT_ROOT / 'data/processed/bank_marketing_clean.csv'
TRAIN_PATH = PROJECT_ROOT / 'data/processed/train_precall.csv'
TEST_PATH = PROJECT_ROOT / 'data/processed/test_precall.csv'
RANDOM_SEED = 42
df = pd.read_csv(DATA_PATH)

In [2]:
X, y = build_precall_features(df)
X_train, X_test, y_train, y_test = stratified_train_test_split(X, y, random_state=RANDOM_SEED)

print(f'特征数：{X.shape[1]}（已排除 y、target_subscription、duration）')
print(f'训练集：{X_train.shape[0]:,} 行；测试集：{X_test.shape[0]:,} 行')
print(f'训练集订购率：{y_train.mean():.2%}；测试集订购率：{y_test.mean():.2%}')
display(X_train.head())

特征数：21（已排除 y、target_subscription、duration）
训练集：32,940 行；测试集：8,236 行
训练集订购率：11.27%；测试集订购率：11.27%


,age,job,marital,education,default,contact,month,day_of_week,campaign,previous,...,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,age_group,previously_contacted,pdays_since_last_contact,has_housing_loan,has_personal_loan
36444,24,technician,single,professional.course,no,cellular,jun,tue,2,1,...,-2.9,92.963,-40.8,1.262,5076.2,"[0.0, 25.0)",0,NaN,0,0
1233,32,blue-collar,married,professional.course,no,telephone,may,thu,1,0,...,1.1,93.994,-36.4,4.855,5191.0,"[25.0, 35.0)",0,NaN,0,0
24104,33,self-employed,single,university.degree,no,telephone,nov,thu,1,0,...,-0.1,93.200,-42.0,4.245,5195.8,"[25.0, 35.0)",0,NaN,0,0
15513,38,entrepreneur,married,basic.9y,no,telephone,jul,fri,4,0,...,1.4,93.918,-42.7,4.957,5228.1,"[35.0, 45.0)",0,NaN,0,1
17912,39,services,married,high.school,no,cellular,jul,tue,4,0,...,1.4,93.918,-42.7,4.961,5228.1,"[35.0, 45.0)",0,NaN,1,1


## 建模预处理器

类别变量采用独热编码，数值变量采用中位数填补与标准化。预处理器仅在训练集上拟合，避免测试集信息泄露。

In [3]:
numeric_features = X_train.select_dtypes(include='number').columns.tolist()
categorical_features = X_train.select_dtypes(include=['object', 'string']).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ('numeric', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), numeric_features),
        ('categorical', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), categorical_features),
    ]
)
X_train_transformed = preprocessor.fit_transform(X_train)
X_test_transformed = preprocessor.transform(X_test)
print(f'数值特征：{len(numeric_features)}；类别特征：{len(categorical_features)}')
print(f'编码后训练特征矩阵：{X_train_transformed.shape}')
assert X_train_transformed.shape[1] == X_test_transformed.shape[1]

数值特征：12；类别特征：9
编码后训练特征矩阵：(32940, 65)


## 保存可复现的数据划分

In [4]:
train_output = X_train.copy()
train_output['target_subscription'] = y_train
test_output = X_test.copy()
test_output['target_subscription'] = y_test
train_output.to_csv(TRAIN_PATH, index=False)
test_output.to_csv(TEST_PATH, index=False)

assert pd.read_csv(TRAIN_PATH).shape == train_output.shape
assert pd.read_csv(TEST_PATH).shape == test_output.shape
print(f'已保存：{TRAIN_PATH.relative_to(PROJECT_ROOT)}')
print(f'已保存：{TEST_PATH.relative_to(PROJECT_ROOT)}')

已保存：data\processed\train_precall.csv
已保存：data\processed\test_precall.csv


## 下一步

使用同一预处理方案训练逻辑回归、随机森林等候选模型，并通过 ROC-AUC、PR-AUC、召回率和业务阈值进行比较。